# Module 3 · Demo — ReAct from Scratch

**From 0 to Agentic AI — DataHack Summit 2026**

In Module 2 the agent "just worked" — `create_agent` hid the loop. This notebook **opens the
hood**: we build that exact loop **by hand**, in plain Python, with no agent framework.

The pattern is **ReAct** — *Reason + Act*:

> **Thought** (reason about what to do) → **Action** (call a tool) → **Observation** (read the
> result) → … repeat until the model has the answer.

> This is a **teaching demo**. The code here is deliberately hand-rolled so the loop is visible.
> In Module 4 we throw it away and rebuild the same loop *properly* with LangGraph.

### What you'll do
1. Define **two** tools — so the model has to *choose* which one to use
2. Read a **tool-call request** from the model (Action)
3. Run the tool and feed the result back (Observation)
4. Wrap it all in a **`while` loop** — the whole agent, ~20 lines
5. Watch it **fail** on purpose, and add the guardrails that fix it

---
## Setup

In [1]:
# Install the workshop stack (Colab). Locally, use `uv sync` instead.
# Version ranges match src/pyproject.toml (the single source of truth).
!pip install -q "langchain>=1.2,<2" "langchain-openai>=1.1,<2"

ERROR: Could not find a version that satisfies the requirement langchain<2,>=1.2 (from versions: 0.0.1, 0.0.2, 0.0.3, 0.0.4, 0.0.5, 0.0.6, 0.0.7, 0.0.8, 0.0.9, 0.0.10, 0.0.11, 0.0.12, 0.0.13, 0.0.14, 0.0.15, 0.0.16, 0.0.17, 0.0.18, 0.0.19, 0.0.20, 0.0.21, 0.0.22, 0.0.23, 0.0.24, 0.0.25, 0.0.26, 0.0.27, 0.0.28, 0.0.29, 0.0.30, 0.0.31, 0.0.32, 0.0.33, 0.0.34, 0.0.35, 0.0.36, 0.0.37, 0.0.38, 0.0.39, 0.0.40, 0.0.41, 0.0.42, 0.0.43, 0.0.44, 0.0.45, 0.0.46, 0.0.47, 0.0.48, 0.0.49, 0.0.50, 0.0.51, 0.0.52, 0.0.53, 0.0.54, 0.0.55, 0.0.56, 0.0.57, 0.0.58, 0.0.59, 0.0.60, 0.0.61, 0.0.63, 0.0.64, 0.0.65, 0.0.66, 0.0.67, 0.0.68, 0.0.69, 0.0.70, 0.0.71, 0.0.72, 0.0.73, 0.0.74, 0.0.75, 0.0.76, 0.0.77, 0.0.78, 0.0.79, 0.0.80, 0.0.81, 0.0.82, 0.0.83, 0.0.84, 0.0.85, 0.0.86, 0.0.87, 0.0.88, 0.0.89, 0.0.90, 0.0.91, 0.0.92, 0.0.93, 0.0.94, 0.0.95, 0.0.96, 0.0.97, 0.0.98, 0.0.99rc0, 0.0.99, 0.0.100, 0.0.101rc0, 0.0.101, 0.0.102rc0, 0.0.102, 0.0.103, 0.0.104, 0.0.105, 0.0.106, 0.0.107, 0.0.108, 0.0.109, 0.0

In [2]:
import os
from getpass import getpass

# Local: load keys from src/.env (walks up to find it). Colab: prompts for missing keys.
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except Exception:
    pass

for key in ["OPENAI_API_KEY"]:
    if not os.environ.get(key):
        os.environ[key] = getpass(f"{key}: ")

---
## Step 1 · Two tools (so there's a choice to make)

"How does the model decide which tool to use?" only means something when there's more than
one. We give it a **calculator** and a tiny **directory lookup**. Each is a plain Python
function — the docstring and type hints are what the model reads to choose.

In [3]:
from langchain_core.tools import tool

@tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression, e.g. '3 * (4 + 5)'."""
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"error: {e}"

@tool
def lookup_employee(name: str) -> str:
    """Look up which team an employee works on, by first name."""
    directory = {"alessandro": "Billing", "sam": "Platform", "dana": "Data"}
    return directory.get(name.lower().strip(), "not found")

tools = [calculator, lookup_employee]
tools_by_name = {t.name: t for t in tools}
print('tools:', list(tools_by_name))

tools: ['calculator', 'lookup_employee']


---
## Step 2 · One turn — the model *requests* an action

We bind the tools to the model and ask a question. The model doesn't answer — it returns a
**tool-call request**: the *Action*. This is the raw material of the loop.

In [4]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4.1-mini", model_provider="openai", temperature=0)
llm_with_tools = llm.bind_tools(tools)

resp = llm_with_tools.invoke("What team is Dana on?")
print('content:   ', repr(resp.content))     # usually empty on a tool turn
print('tool_calls:', resp.tool_calls)         # the Action: {name, args}

content:    ''
tool_calls: [{'name': 'lookup_employee', 'args': {'name': 'Dana'}, 'id': 'call_pIq9crQ64eFMwrTiPeJlfxVr', 'type': 'tool_call'}]


### Run the action, get the observation
The model only *asked*. **We** run the tool and produce the **Observation** — a `ToolMessage`
carrying the result, tagged with the request's `id` so the model knows which call it answers.

In [5]:
from langchain_core.messages import ToolMessage

call = resp.tool_calls[0]
chosen = tools_by_name[call['name']]
observation = chosen.invoke(call['args'])
print('observation:', observation)

tool_msg = ToolMessage(content=str(observation), tool_call_id=call['id'])
print(tool_msg)

observation: Data
content='Data' tool_call_id='call_pIq9crQ64eFMwrTiPeJlfxVr'


That's **one turn** of ReAct: Thought (implicit in the model) → Action (`tool_calls`) →
Observation (`ToolMessage`). Now we just... keep going until there's no more action to take.

---
## Step 3 · The loop, by hand

The whole agent is a `while` loop over a growing **message history**:

- ask the model,
- if it requested tools → run them, append the observations, **loop**,
- if it didn't → that's the final answer, **stop**.

Note the **step cap** (`max_steps`) — without it, a confused model can loop forever.

In [6]:
from langchain_core.messages import HumanMessage

def run_agent(question: str, max_steps: int = 5, verbose: bool = True):
    messages = [HumanMessage(content=question)]

    for step in range(max_steps):
        ai = llm_with_tools.invoke(messages)
        messages.append(ai)

        # No tool call -> the model is done. This is the Final Answer.
        if not ai.tool_calls:
            if verbose:
                print(f"[step {step}] FINAL: {ai.content}")
            return ai.content

        # Otherwise, run each requested tool (Action -> Observation).
        for call in ai.tool_calls:
            if verbose:
                print(f"[step {step}] ACTION: {call['name']}({call['args']})")
            tool = tools_by_name.get(call['name'])
            result = tool.invoke(call['args']) if tool else f"unknown tool: {call['name']}"
            if verbose:
                print(f"[step {step}] OBSERVE: {result}")
            messages.append(ToolMessage(content=str(result), tool_call_id=call['id']))

    return "stopped: hit max_steps without finishing"

### Run it — a single-tool question

In [7]:
run_agent("What team is Dana on?")

[step 0] ACTION: lookup_employee({'name': 'Dana'})
[step 0] OBSERVE: Data
[step 1] FINAL: Dana is on the Data team.


'Dana is on the Data team.'

### Run it — a question that needs the *other* tool
Same loop, different tool. The model **chooses** based on the docstrings we wrote.

In [8]:
run_agent("What is 47 * 19?")

[step 0] ACTION: calculator({'expression': '47 * 19'})
[step 0] OBSERVE: 893
[step 1] FINAL: 47 multiplied by 19 is 893.


'47 multiplied by 19 is 893.'

### Run it — a question that needs **both** tools, in sequence
Watch the loop take more than one turn: it reasons, acts, observes, then reasons again.
This is why an agent is a *loop*, not a single call.

In [9]:
q = ("Dana is on a team of 4 and Sam is on a team of 6. ""Look up which teams they are on, then use the calculator ""to total the two team sizes.")
run_agent(q)

[step 0] ACTION: lookup_employee({'name': 'Dana'})
[step 0] OBSERVE: Data
[step 0] ACTION: lookup_employee({'name': 'Sam'})
[step 0] OBSERVE: Platform
[step 1] ACTION: calculator({'expression': '4 + 6'})
[step 1] OBSERVE: 10
[step 2] FINAL: Dana is on the Data team, and Sam is on the Platform team. The total number of people on their teams combined is 10.


'Dana is on the Data team, and Sam is on the Platform team. The total number of people on their teams combined is 10.'

---
## Step 4 · Watch it break

Hand-rolled loops fail in predictable ways. Two we already guarded against — and one we didn't.

### 🔁 Runaway steps → stopped by the cap
A multi-tool task needs several turns. If we set `max_steps` **too low**, the loop is cut off
before it finishes — and returns our safety message instead of running forever. That same cap
is what protects you when a *confused* model would otherwise loop without end.

In [10]:
# This task needs 3 lookups + a sum (≈4 turns), but we only allow 2 steps:
q = ("Look up the teams for Dana, Sam, and Alessandro, then use the "
     "calculator to report how many distinct teams that is.")
print(run_agent(q, max_steps=2))

[step 0] ACTION: lookup_employee({'name': 'Dana'})
[step 0] OBSERVE: Data
[step 0] ACTION: lookup_employee({'name': 'Sam'})
[step 0] OBSERVE: Platform
[step 0] ACTION: lookup_employee({'name': 'Alessandro'})
[step 0] OBSERVE: Billing
[step 1] ACTION: calculator({'expression': '3'})
[step 1] OBSERVE: 3
stopped: hit max_steps without finishing


### 👻 Hallucinated / mis-used tool
The model can call the *wrong* tool, or pass junk arguments. Here a name that isn't in the
directory just yields `not found` — but notice the agent still has to **decide what to do**
with a useless observation. Robust tools return readable errors instead of crashing.

In [11]:
run_agent("What team is Priya on?")

[step 0] ACTION: lookup_employee({'name': 'Priya'})
[step 0] OBSERVE: not found
[step 1] FINAL: I couldn't find any information about an employee named Priya. Could you please provide more details or check the spelling?


"I couldn't find any information about an employee named Priya. Could you please provide more details or check the spelling?"

> **The failure modes to remember:** looping (cap the steps), hallucinated actions (validate the tool name), and tool misuse (validate the args / return readable errors). A production loop needs all three guards — which is a big part of why we move to a framework next.

---
## Key takeaways
- An agent is a **loop**: Thought → Action → Observation → … → Final Answer.
- The model only **requests** actions; *your* code runs them and feeds back observations.
- With ≥2 tools, tool **selection** is driven by the names, docstrings, and arg schemas you write.
- Hand-rolled loops need guards: **step caps**, **tool-name validation**, **arg/error handling**.

➡️ **Next (Module 4):** we rebuild this exact loop with **LangGraph** — the guards, state, and
routing become first-class instead of hand-written. That's where the real app build begins.